---
jupytext:
  text_representation:
    format_name: myst
kernelspec:
  display_name: Python 3
  language: python
  name: python3
---
# Lesson 4 Ã¢â‚¬â€ Dates, Resampling & Water Years

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mr-pablinho/drops-course/blob/main/book/01-python-for-hydrology/04-dates-resampling.ipynb)

**Learning objectives**  
By the end of this lesson you will be able to:
- [ ] Navigate a Pandas `DatetimeIndex` and extract time components (year, month, day-of-year)
- [ ] Resample daily data to weekly, monthly, and annual frequencies using `df.resample()`
- [ ] Define and label **USGS water years** (October 1 Ã¢â‚¬â€œ September 30) in a time series
- [ ] Compute annual peak flows by water year and identify trends

In [ ]:
# Install course dependencies when running on Google Colab
import sys
if 'google.colab' in sys.modules:
    import subprocess
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        '-r', 'https://raw.githubusercontent.com/mr-pablinho/drops-course/main/requirements-colab.txt'
    ], check=True)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

In [ ]:
CI = os.environ.get('HYDRO_ML_CI', '0') == '1'

if CI:
    df_raw = pd.read_parquet('../../data/samples/usgs_06803495_daily_2000_2015.parquet')
else:
    import dataretrieval.nwis as nwis
    raw, _ = nwis.get_dv(
        sites='06803495',
        parameterCd='00060',
        start='2000-01-01',
        end='2015-12-31',
    )
    df_raw = raw.reset_index()

df = (
    df_raw
    .rename(columns={'00060_Mean': 'discharge_cfs'})
    .assign(datetime=lambda d: pd.to_datetime(d['datetime']))
    .set_index('datetime')
    .sort_index()
    [['discharge_cfs']]
    .pipe(lambda d: d[d['discharge_cfs'] > 0])
    .dropna()
)

print(f'Loaded {len(df):,} daily records from {df.index.min().date()} to {df.index.max().date()}')

## Introduction

Hydrology is inherently a time science. Floods peak at specific hours, snowmelt
follows the calendar, and drought accumulates over months to years. Pandas
`DatetimeIndex` makes time-aware operations Ã¢â‚¬â€ slicing, resampling, shifting,
and year-based grouping Ã¢â‚¬â€ first-class operations that run without manual
indexing arithmetic.

The single most consequential date convention in US hydrology is the
**water year**: instead of the calendar year (JanÃ¢â‚¬â€œDec), the USGS defines
a water year as **October 1 through September 30**. Water Year 2011 ran
from 1 Oct 2010 to 30 Sep 2011 and contains the peak of the Missouri
River Flood in its entirety Ã¢â‚¬â€ which is exactly why this convention exists.

## 1. Anatomy of a DatetimeIndex

A Pandas `DatetimeIndex` stores timestamps and exposes calendar attributes
as vectorised array accessors. No loops required.

In [ ]:
idx = df.index
print(f'Type     : {type(idx).__name__}')
print(f'Length   : {len(idx):,}')
print(f'Freq     : {idx.freq}  (None = irregular Ã¢â‚¬â€ gaps possible)')
print(f'First    : {idx[0]}')
print(f'Last     : {idx[-1]}')
print()

# Accessor arrays Ã¢â‚¬â€ all vectorised, no for-loops needed
years      = idx.year          # integer year
months     = idx.month         # 1-12
doy        = idx.day_of_year   # 1-366
weekday    = idx.day_of_week   # 0=Monday Ã¢â‚¬Â¦ 6=Sunday

print(f'Unique years in record : {sorted(set(years))}')
print(f'Max day-of-year present: {doy.max()}')

# Gap detection: any missing calendar days?
full_range = pd.date_range(idx.min(), idx.max(), freq='D')
missing = full_range.difference(idx)
print(f'\nCalendar days in range : {len(full_range):,}')
print(f'Days in our record     : {len(idx):,}')
print(f'Missing days           : {len(missing)}')
if len(missing):
    print(f'  First gap: {missing[0].date()}')

## 2. Resampling: Changing the Time Step

`DataFrame.resample(rule)` groups consecutive observations into bins of
the specified frequency and returns a `Resampler` to which you chain an
aggregation (`.mean()`, `.max()`, `.sum()`, etc.).

Commonly used frequency strings in hydrology:

| String | Meaning |
|---|---|
| `'W'` | Calendar week (Sunday) |
| `'ME'` | Month-end |
| `'QE-SEP'` | Quarter ending in September (water-year quarters) |
| `'YE'` | Calendar year-end (December 31) |
| `'YE-SEP'` | Year ending September 30 (water year) |

In [ ]:
weekly  = df.resample('W')['discharge_cfs'].mean()
monthly = df.resample('ME')['discharge_cfs'].mean()
annual  = df.resample('YE')['discharge_cfs'].mean()

print(f'Daily  records: {len(df):>5,}')
print(f'Weekly records: {len(weekly):>5,}')
print(f'Monthly records:{len(monthly):>5,}')
print(f'Annual records: {len(annual):>5,}')

fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=False)

for ax, data, label, color in zip(
    axes,
    [weekly, monthly, annual],
    ['Weekly mean', 'Monthly mean', 'Annual mean'],
    ['steelblue', 'royalblue', 'navy'],
):
    ax.bar(data.index, data / 1_000, width=data.index.to_series().diff().dt.days.fillna(7),
           color=color, alpha=0.65, align='edge')
    ax.set_ylabel('kcfs', fontsize=10)
    ax.set_title(label, fontsize=10)
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:.0f}k'))

axes[-1].set_xlabel('Date', fontsize=11)
plt.suptitle('Missouri River Ã¢â‚¬â€ Resampled Discharge at Three Frequencies', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## 3. Water Years

The **USGS water year** starts on October 1 and ends September 30.
Water Year *Y* covers October 1 of calendar year *Y-1* through September 30 of year *Y*.

To label each day with its water year, we shift the month by three months:
if the shifted month (i.e., `month + 3`) rolls into a new calendar year,
the water year is that new year. The one-line Pandas idiom is:

```python
water_year = (df.index + pd.DateOffset(months=3)).year
```

This works because shifting October (month 10) by +3 gives January
of the *next* calendar year Ã¢â‚¬â€ which is the correct water-year label.

In [ ]:
df = df.copy()
df['water_year'] = (df.index + pd.DateOffset(months=3)).year

# Verify the boundary: 30-Sep and 01-Oct of 2010
boundary = df.loc['2010-09-28':'2010-10-03', 'water_year']
print('Water-year label around 1 Oct 2010:')
print(boundary.to_string())
print()

# Annual peak discharge by water year
wy_peak = df.groupby('water_year')['discharge_cfs'].max()
print('Annual peak discharge by water year:')
for wy, val in wy_peak.items():
    flag = ' Ã¢â€ Â WY2011 flood' if wy == 2011 else ''
    print(f'  WY{wy}: {val:>12,.0f} cfs{flag}')

## 4. Plotting Water-Year Peak Flows

The water-year peak series is the input to **flood frequency analysis**
(Lesson 6 will cover Gumbel / GEV fits). Here we simply visualise the
series and mark the long-term median to give a sense of variability.

In [ ]:
median_peak = wy_peak.median()

colors = ['crimson' if wy == 2011 else 'steelblue' for wy in wy_peak.index]

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(wy_peak.index, wy_peak / 1_000, color=colors, alpha=0.8)
ax.axhline(median_peak / 1_000, color='navy', ls='--', lw=1.8,
           label=f'Median annual peak: {median_peak:,.0f} cfs')

# Annotate the 2011 flood bar
ax.annotate(
    f'WY2011\n{wy_peak[2011] / 1_000:.0f} kcfs',
    xy=(2011, wy_peak[2011] / 1_000),
    xytext=(2012.5, wy_peak[2011] / 1_000 * 0.88),
    arrowprops=dict(arrowstyle='->', color='darkred'),
    color='darkred', fontsize=9,
)

ax.set_xlabel('Water Year', fontsize=11)
ax.set_ylabel('Annual Peak Discharge (kcfs)', fontsize=11)
ax.set_title('Missouri River at Omaha Ã¢â‚¬â€ Annual Peak Discharge by Water Year', fontsize=11)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:.0f}k'))
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

# Summary statistics on the peak series
print(f'WY peak series Ã¢â‚¬â€ median : {median_peak:>12,.0f} cfs')
print(f'WY peak series Ã¢â‚¬â€ mean   : {wy_peak.mean():>12,.0f} cfs')
print(f'WY peak series Ã¢â‚¬â€ cv     : {wy_peak.std() / wy_peak.mean():.2f}  (high = more variable)')

## 5. Monthly Resampling with `resample('ME')`

The `resample()` API is not limited to means: `.max()`, `.min()`, `.sum()`,
and `.agg({'col': func})` all work. Below we combine monthly mean and
monthly maximum on the same axes to show how much of the month's discharge
is driven by short-duration peaks.

In [ ]:
monthly_stats = df['discharge_cfs'].resample('ME').agg(
    mean_cfs='mean',
    max_cfs='max',
)

# Focus on 2008-2012 to see the flood in context
sub = monthly_stats['2008':'2012']

fig, ax = plt.subplots(figsize=(13, 4))
ax.bar(sub.index, sub['max_cfs'] / 1_000,
       width=20, color='lightcoral', alpha=0.6, label='Monthly max')
ax.plot(sub.index, sub['mean_cfs'] / 1_000,
        'o-', color='navy', lw=1.8, ms=5, label='Monthly mean')

ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Discharge (kcfs)', fontsize=11)
ax.set_title('Missouri River Ã¢â‚¬â€ Monthly Mean vs Monthly Max Discharge 2008Ã¢â‚¬â€œ2012', fontsize=11)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:.0f}k'))
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## Key Takeaways

- A Pandas `DatetimeIndex` exposes `.year`, `.month`, `.day_of_year`,
  and other calendar attributes as vectorised arrays Ã¢â‚¬â€ no loops needed
  for time-based feature engineering.
- `resample(rule).agg()` changes the temporal resolution in one call;
  the offset alias `'YE-SEP'` produces water-year end-of-period statistics
  that align with USGS reporting conventions.
- The **water year** (Oct 1 Ã¢â‚¬â€œ Sep 30) is the standard hydrological year in
  the USA; labelling with `(index + DateOffset(months=3)).year` correctly
  assigns OctoberÃ¢â‚¬â€œDecember to the *following* year number.
- Annual peak-flow series are the foundation of flood frequency analysis;
  the coefficient of variation (CV = ÃÆ’/ÃŽÂ¼) summarises how variable a gauge's
  floods are from year to year.

## Exercise

**Try it yourself:**  
Compute the **water-year total volume** (in million acre-feet, MAF) for each
water year in the record. Plot a bar chart of MAF by water year.

One acre-foot = 1 cfs Ãƒâ€” 1.9835 days, so to convert daily mean cfs to acre-feet:
`volume_af = discharge_cfs * 1.9835`.

Then sum within each water year using `groupby('water_year')['volume_af'].sum()`.

*Which water year had the largest total volume, and how does it compare to the
long-term median?*

In [ ]:
# Solution Ã¢â‚¬â€ try on your own first!
df2 = df.copy()
df2['volume_af'] = df2['discharge_cfs'] * 1.9835

wy_vol_maf = df2.groupby('water_year')['volume_af'].sum() / 1e6
median_maf = wy_vol_maf.median()

colors = ['crimson' if wy == 2011 else 'steelblue' for wy in wy_vol_maf.index]

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(wy_vol_maf.index, wy_vol_maf, color=colors, alpha=0.8)
ax.axhline(median_maf, color='navy', ls='--', lw=1.8,
           label=f'Median: {median_maf:.1f} MAF')

ax.set_xlabel('Water Year', fontsize=11)
ax.set_ylabel('Total Volume (MAF)', fontsize=11)
ax.set_title('Missouri River at Omaha Ã¢â‚¬â€ Water-Year Total Volume', fontsize=11)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

top = wy_vol_maf.idxmax()
print(f'Largest water year : WY{top}  ({wy_vol_maf[top]:.1f} MAF)')
print(f'Long-term median   : {median_maf:.1f} MAF')
print(f'Ratio (flood/median): {wy_vol_maf[top] / median_maf:.2f}x')